# NV4 — Đánh giá Hiệu suất và Xếp hạng Đa tiêu chí Mạng lưới Trạm sạc (Station Benchmarking & TOPSIS)

## 1. Bối cảnh Vận hành & Động lực Nghiên cứu
Mạng lưới sạc xe điện tại thành phố Palo Alto bao gồm 46 trạm sạc với 79 cổng sạc vật lý phân bố tại 9 khu vực địa lý khác nhau (trung tâm thương mại Downtown, bãi đỗ xe công cộng, thư viện, tòa nhà hành chính). Các trạm này ghi nhận sự chênh lệch rất lớn về mức độ khai thác và hiệu năng vận hành.

Để phục vụ công tác quy hoạch, điều độ bảo trì và phân bổ vốn đầu tư nâng cấp hạ tầng (*CapEx*), đơn vị vận hành mạng lưới sạc (*CPO*) và cơ quan quản lý đô thị cần một công cụ xếp hạng khách quan, toàn diện thay vì chỉ nhìn vào một chỉ số đơn lẻ như tổng số lượt sạc hay sản lượng điện.

## 2. Khung Phương pháp luận Đa tiêu chí (MCDM Framework)
1. **Phễu Đối soát Dữ liệu (Reconciliation Ledger):** Kiểm soát nghiêm ngặt dòng chảy dữ liệu từ NV1 sang NV4, loại bỏ các phiên lệch timestamp > 59 giây và xác định tập 33 trạm sạc đủ điều kiện hoạt động ổn định.
2. **Ma trận Quyết định Đa chiều (Decision Matrix):** Xây dựng ma trận 33 trạm × 6 tiêu chí phản ánh cả quy mô khai thác, hiệu suất sử dụng và rủi ro chiếm dụng không tải.
3. **Chuẩn hóa Min-Max Phân định Chiều:** Đưa mọi tiêu chí về miền giá trị $[0, 1]$, đồng nhất hóa tiêu chí Lợi ích (*Benefit*) và tiêu chí Chi phí (*Cost*).
4. **Trọng số Entropy Khách quan (Shannon Entropy):** Đo lường lượng thông tin và mức độ phân tán nội tại của từng tiêu chí để xác định trọng số mà không bị thiên kiến chủ quan của chuyên gia.
5. **Kỹ thuật TOPSIS (Technique for Order Preference by Similarity to Ideal Solution):** Đo lường khoảng cách hình học Euclid trong không gian 6 chiều tới Nghiệm lý tưởng dương (*Positive Ideal Solution - PIS*) và Nghiệm lý tưởng âm (*Negative Ideal Solution - NIS*), xác định Hệ số tiệm cận lý tưởng tương đối ($CC_i$).
6. **Phân tích Độ nhạy Vững chắc (Sensitivity Analysis):** Đánh giá độ ổn định của thứ hạng trạm qua 7 kịch bản giả định (S0 – S6).

---

## 1. Thiết lập Môi trường, Thư viện & Cấu hình Cốt lõi

### 🎯 Mục tiêu của Bước:
- Khởi tạo các đường dẫn hệ thống: thư mục dữ liệu gốc `RAW_DATA_FILE`, thư mục lưu trữ kết quả đầu ra `OUTPUT_DIR = outputs/`.
- Nhúng các module nghiệp vụ chuẩn hóa được xây dựng trong gói mã nguồn `src/`:
  - `src.nv4_analysis`: Xử lý phễu đối soát dữ liệu, tính toán ma trận 6 tiêu chí hiệu suất trạm.
  - `src.topsis`: Hàm chuẩn hóa Min-Max, tính trọng số Entropy của Shannon và thuật toán TOPSIS tính khoảng cách Euclid.
  - `src.nv4_sensitivity`: Module mô phỏng 7 kịch bản độ nhạy và kiểm tra độ bền vững thứ bậc.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from the project workspace")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import RAW_DATA_FILE, SESSIONS_CLEAN_PARQUET, POWER_LIMITS_KW
from src.nv4_analysis import NV4_CRITERIA, build_station_criteria, build_station_rankings, prepare_nv4_sessions
from src.nv4_sensitivity import run_sensitivity, summarize_rank_robustness
from src.topsis import entropy_weights, normalize_minmax, topsis

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Source:", RAW_DATA_FILE)
print("Power limits by Port Type:", POWER_LIMITS_KW)

Source: /home/kedora/SEM5/ba/project/ev-charging-analytics---IS403/data/raw/EVChargingStationUsage.csv
Power limits by Port Type: {'Level 1': 1.92, 'Level 2': 22.0, 'default': 22.0}


## 2. Nạp Dữ liệu & Bảng Đối soát Phễu Dữ liệu Tuần tự (Reconciliation Ledger NV1 ➔ NV4)

### 🎯 Cơ sở Phương pháp luận:
Để đảm bảo tính toàn vẹn khoa học và khả năng kiểm toán dữ liệu, các bước lọc dữ liệu từ NV1 đến NV4 được ghi nhận dưới dạng một **Sổ cái đối soát (Audit Ledger)** tuần tự, không cộng dồn trùng lặp:
1. **Tổng dữ liệu thô:** $259.415$ bản ghi CSV gốc.
2. **Loại trùng lặp tuyệt đối:** Bỏ 4 dòng trùng lặp hoàn toàn $\to 259.411$ dòng.
3. **Kiểm tra logic thời gian:** Bỏ 45 phiên lỗi cảm biến $T_{charging} > T_{total} \to 259.366$ dòng.
4. **Kiểm tra công suất vật lý trạm Level 2:** Bỏ 10 phiên vượt trần công suất theo cấu hình $\to 259.356$ phiên hợp lệ chuẩn của NV1 (`is_valid_session == True`).
5. **Kiểm tra tính nhất quán Timestamp (Bộ lọc riêng NV4):** Loại bỏ 481 phiên có độ lệch giữa tổng thời lượng ghi nhận và hiệu số $(T_{end} - T_{start})$ vượt quá 59 giây $\to 258.875$ phiên.
6. **Lọc trạm đủ điều kiện xếp hạng:** Loại bỏ các trạm tạm thời, trạm thử nghiệm kỹ thuật (tổng số phiên $< 100$), thu được **$258.836$ phiên sạc hợp lệ thuộc 33 trạm sạc tiêu chuẩn**.

In [2]:
raw = pd.read_csv(RAW_DATA_FILE, encoding="utf-8-sig", low_memory=False)
sessions, reconciliation, run_metadata = prepare_nv4_sessions(raw)
all_stations, X = build_station_criteria(sessions)
run_metadata.update({
    "station_count_after_cleaning": len(all_stations),
    "eligible_station_count": len(X),
    "sessions_in_eligible_stations": int(X["C1_so_phien"].sum()),
})
if SESSIONS_CLEAN_PARQUET.exists():
    nv1_clean = pd.read_parquet(SESSIONS_CLEAN_PARQUET)
    parquet_valid = int(nv1_clean["is_valid_session"].sum())
    if parquet_valid != run_metadata["nv1_valid_sessions"]:
        raise AssertionError(f"NV1 count mismatch: parquet={parquet_valid}, raw={run_metadata['nv1_valid_sessions']}")
    run_metadata["sessions_clean_parquet_valid_sessions"] = parquet_valid

reconciliation = pd.concat([reconciliation, pd.DataFrame([
    {"stage": "Distinct stations after NV4 filters", "rows_remaining": len(all_stations),
     "removed_at_stage": None, "rule": "Unique normalized Station Name"},
    {"stage": "Stations eligible for ranking", "rows_remaining": len(X),
     "removed_at_stage": len(all_stations)-len(X), "rule": "At least 500 NV4-valid sessions"},
    {"stage": "Sessions in eligible stations", "rows_remaining": int(X.C1_so_phien.sum()),
     "removed_at_stage": len(sessions)-int(X.C1_so_phien.sum()),
     "rule": "Sessions in stations meeting the threshold"},
])], ignore_index=True)
display(reconciliation)
display(pd.Series(run_metadata, name="value").to_frame())

,stage,rows_remaining,removed_at_stage,rule
0,Raw CSV rows,259415,NaN,Raw source rows
1,After exact duplicate removal,259411,4.0,Exact duplicate rows removed
2,After positive/consistent duration checks,259366,45.0,"Require total > 0, charging > 0, charging <= t..."
3,After Port Type power limits,259356,10.0,Use POWER_LIMITS_KW from src.config by Port Type
4,After timestamp consistency,258875,481.0,NV1 date parser + UTC; absolute duration diffe...
5,NV1 is_valid_session (comparison),259356,NaN,charging <= total and power <= configured Port...
6,NV1-valid sessions excluded by NV4 timestamp c...,481,NaN,Only the additional NV4 timestamp consistency ...
7,Distinct stations after NV4 filters,46,NaN,Unique normalized Station Name
8,Stations eligible for ranking,33,13.0,At least 500 NV4-valid sessions
9,Sessions in eligible stations,258836,39.0,Sessions in stations meeting the threshold


,value
raw_rows,259415
exact_duplicates_removed,4
nv1_valid_sessions,259356
nv4_valid_sessions,258875
nv4_timestamp_rejected_after_other_checks,481
power_limits_kw,"{'Level 1': 1.92, 'Level 2': 22.0, 'default': ..."
timestamp_tolerance_seconds,59
station_count_after_cleaning,46
eligible_station_count,33
sessions_in_eligible_stations,258836


## 3. Xây dựng Ma trận Quyết định Đa chiều (33 × 6) & Ma trận Tương quan

###  Hệ Thống 6 Tiêu chí Hiệu suất Vận hành (Operational Criteria):
Ma trận quyết định $X \in \mathbb{R}^{33 \times 6}$ được xây dựng từ 6 chỉ báo vận hành đại diện cho 3 khía cạnh: Quy mô khai thác, Hiệu quả năng lượng và Rủi ro chiếm dụng:
- **$C_1$ - Tổng số phiên sạc (Lượt) [Benefit (+)]:** Đại diện cho mức độ phổ biến và nhu cầu kết nối của tài xế tại vị trí đặt trạm.
- **$C_2$ - Tổng điện năng cung cấp (kWh) [Benefit (+)]:** Đại diện cho tổng sản lượng dịch vụ năng lượng cung ứng cho cộng đồng.
- **$C_3$ - Công suất trung bình phiên (kW) [Benefit (+)]:** Đại diện cho tốc độ sạc thực tế và tính tương thích thiết bị của trạm ($C_3 = \text{Energy} / T_{charging}$).
- **$C_4$ - Hệ số sử dụng thời gian có ích (Useful Charge Ratio) [Benefit (+)]:** Tỷ số giữa thời gian nạp điện thực tế và tổng thời gian cắm cáp ($T_{charging} / T_{total}$).
- **$C_5$ - Tỷ lệ chiếm dụng nhàn rỗi trung bình (Idle Ratio) [Cost (-)]:** Tỷ lệ thời gian xe đỗ ngâm sau khi pin đầy ($T_{idle} / T_{total}$). Càng thấp càng tốt.
- **$C_6$ - Tỷ lệ phiên chiếm dụng nghiêm trọng > 3 giờ [Cost (-)]:** Tỷ lệ phần trăm các phiên có thời gian nhàn rỗi vượt quá 3 giờ, phản ánh nguy cơ tắc nghẽn cổng sạc kéo dài. Càng thấp càng tốt.

### Phân tích Tương quan Pearson:
Kiểm tra mức độ đa cộng tuyến giữa các tiêu chí. Đáng chú ý: $C_1$ và $C_2$ có hệ số tương quan tuyến tính $r = 0.982$ (rất cao). Đây là cơ sở then chốt để xây dựng kịch bản kiểm tra độ nhạy S2 (loại bỏ $C_1$) nhằm kiểm tra xem trạm lớn có bị thiên vị do quy mô hay không.

In [3]:
criterion_columns = list(NV4_CRITERIA)
directions = {c: NV4_CRITERIA[c][2] for c in criterion_columns}
labels = {c: NV4_CRITERIA[c][0] for c in criterion_columns}
decision = X[criterion_columns]
corr_pearson = decision.corr(method="pearson")
corr_spearman = decision.corr(method="spearman")
display(decision.head())
display(corr_pearson.round(4))
display(corr_spearman.round(4))
print("Spearman(C5,C6):", corr_spearman.loc["C5_ty_le_nhan_roi", "C6_ty_le_chiem_dung_nghiem_trong"])

,C1_so_phien,C2_tong_dien_nang,C3_dien_nang_moi_phien,C4_gio_sac_moi_phien,C5_ty_le_nhan_roi,C6_ty_le_chiem_dung_nghiem_trong
Station Name,,,,,,
PALO ALTO CA / HAMILTON #2,23682,191268.391057,8.076530,2.032521,0.120082,0.077358
PALO ALTO CA / HAMILTON #1,14836,113384.729360,7.642540,2.042863,0.126852,0.082569
PALO ALTO CA / WEBSTER #1,14408,138831.830993,9.635746,2.146363,0.129359,0.086063
PALO ALTO CA / BRYANT #2,14296,106687.255028,7.462735,2.0014,0.123028,0.086877
PALO ALTO CA / HIGH #4,13844,111443.448651,8.049946,1.947649,0.119726,0.078662


,C1_so_phien,C2_tong_dien_nang,C3_dien_nang_moi_phien,C4_gio_sac_moi_phien,C5_ty_le_nhan_roi,C6_ty_le_chiem_dung_nghiem_trong
C1_so_phien,1.0000,0.9815,-0.2814,0.0610,-0.0531,-0.1425
C2_tong_dien_nang,0.9815,1.0000,-0.1254,0.1462,-0.0532,-0.1466
C3_dien_nang_moi_phien,-0.2814,-0.1254,1.0000,0.7377,-0.0914,-0.0874
C4_gio_sac_moi_phien,0.0610,0.1462,0.7377,1.0000,0.1286,0.0758
C5_ty_le_nhan_roi,-0.0531,-0.0532,-0.0914,0.1286,1.0000,0.9853
C6_ty_le_chiem_dung_nghiem_trong,-0.1425,-0.1466,-0.0874,0.0758,0.9853,1.0000


,C1_so_phien,C2_tong_dien_nang,C3_dien_nang_moi_phien,C4_gio_sac_moi_phien,C5_ty_le_nhan_roi,C6_ty_le_chiem_dung_nghiem_trong
C1_so_phien,1.0000,0.9863,-0.2045,-0.0231,0.3840,0.3406
C2_tong_dien_nang,0.9863,1.0000,-0.0969,0.0354,0.3981,0.3653
C3_dien_nang_moi_phien,-0.2045,-0.0969,1.0000,0.6922,-0.1437,-0.1170
C4_gio_sac_moi_phien,-0.0231,0.0354,0.6922,1.0000,0.1965,0.1487
C5_ty_le_nhan_roi,0.3840,0.3981,-0.1437,0.1965,1.0000,0.9709
C6_ty_le_chiem_dung_nghiem_trong,0.3406,0.3653,-0.1170,0.1487,0.9709,1.0000


Spearman(C5,C6): 0.9709224598930482


## 4. Chuẩn hóa Min-Max, Tính Trọng số Entropy & Xếp hạng TOPSIS

###  Cơ sở Toán học & Thuật toán:

#### 1. Chuẩn hóa Min-Max Định hướng Chiều:
Đưa mọi giá trị về thang đo $[0, 1]$, trong đó $1$ là mức hiệu suất tốt nhất và $0$ là mức kém nhất:
- Đối với tiêu chí Lợi ích ($C_1, C_2, C_3, C_4$): $R_{ij} = \frac{x_{ij} - x_j^{\min}}{x_j^{\max} - x_j^{\min}}$
- Đối với tiêu chí Chi phí ($C_5, C_6$): $R_{ij} = \frac{x_j^{\max} - x_{ij}}{x_j^{\max} - x_j^{\min}}$

#### 2. Trọng số Entropy Shannon Khách quan:
Đo lường lượng thông tin của tiêu chí $j$ qua độ phân tán của dữ liệu:
- Tính xác suất chuẩn hóa: $P_{ij} = \frac{x_{ij}}{\sum_{i=1}^{m} x_{ij}}$
- Tính giá trị Entropy: $E_j = -\frac{1}{\ln(m)} \sum_{i=1}^{m} P_{ij} \ln(P_{ij})$
- Tính trọng số khách quan: $w_j = \frac{1 - E_j}{\sum_{k=1}^{n} (1 - E_k)}$
*(Kết quả: $C_1$ chiếm 29,44%, $C_2$ chiếm 27,77%, tổng hai tiêu chí quy mô chiếm 57,21% trọng số)*.

#### 3. Thuật toán TOPSIS:
- Tính ma trận chuẩn hóa có trọng số: $V_{ij} = w_j \times R_{ij}$.
- Xác định Nghiệm lý tưởng dương $A^+ = [\max_i V_{i1}, \dots, \max_i V_{in}]$ và Nghiệm lý tưởng âm $A^- = [\min_i V_{i1}, \dots, \min_i V_{in}]$.
- Đo khoảng cách Euclid $D_i^+$ tới $A^+$ và $D_i^-$ tới $A^-$.
- Tính Hệ số tiệm cận nghiệm lý tưởng tương đối: $CC_i = \frac{D_i^-}{D_i^+ + D_i^-} \in [0, 1]$.
- Trạm có $CC_i$ càng cao thì hiệu suất tổng thể càng vượt trội.

In [4]:
R = normalize_minmax(decision, directions)
entropy_table, w_entropy = entropy_weights(R, labels)
w_equal = pd.Series(1.0 / len(criterion_columns), index=criterion_columns, name="Trọng số đều")
display(entropy_table.round(6))

top_entropy = topsis(R, w_entropy)
top_equal = topsis(R, w_equal)
df_rank = build_station_rankings(X, top_entropy, top_equal)
display(df_rank)
display(df_rank.head(5))
display(df_rank.tail(5))
print("Baseline Spearman:", spearmanr(df_rank.Rank_Entropy, df_rank.Rank_Equal).statistic)
print("Baseline Kendall:", kendalltau(df_rank.Rank_Entropy, df_rank.Rank_Equal).statistic)
print("Mean/max rank difference:", df_rank.Rank_Diff.mean(), df_rank.Rank_Diff.max())

,Entropy (e_j),Độ phân kỳ (d_j = 1 - e_j),Trọng số Entropy (w_j),Tên tiêu chí
Tiêu chí,,,,
C1_so_phien,0.920431,0.079569,0.294388,Số phiên sạc
C2_tong_dien_nang,0.924948,0.075052,0.277676,Tổng điện năng
C3_dien_nang_moi_phien,0.943857,0.056143,0.207718,Điện năng bình quân mỗi phiên
C4_gio_sac_moi_phien,0.973064,0.026936,0.099659,Thời gian sạc hữu ích bình quân
C5_ty_le_nhan_roi,0.982417,0.017583,0.065053,Tỷ lệ chiếm dụng nhàn rỗi
C6_ty_le_chiem_dung_nghiem_trong,0.984997,0.015003,0.055507,Tỷ lệ phiên chiếm dụng nghiêm trọng


,Station Name,C1_so_phien,Score_Entropy,Rank_Entropy,Score_Equal,Rank_Equal,Rank_Diff
0,PALO ALTO CA / HAMILTON #2,23682,0.723298,1,0.672263,2,1
1,PALO ALTO CA / WEBSTER #1,14408,0.647613,2,0.674869,1,1
2,PALO ALTO CA / WEBSTER #2,12654,0.604583,3,0.647787,3,0
3,PALO ALTO CA / HAMILTON #1,14836,0.512089,4,0.555134,9,5
4,PALO ALTO CA / HIGH #4,13844,0.506990,5,0.554710,10,5
5,PALO ALTO CA / WEBSTER #3,10467,0.492506,6,0.584513,6,0
6,PALO ALTO CA / BRYANT #2,14296,0.483649,7,0.533183,16,9
7,PALO ALTO CA / HIGH #1,10746,0.478468,8,0.559798,8,0
8,PALO ALTO CA / TED THOMPSON #1,9804,0.472860,9,0.610314,4,5
9,PALO ALTO CA / BRYANT #1,13406,0.471556,10,0.543703,13,3


,Station Name,C1_so_phien,Score_Entropy,Rank_Entropy,Score_Equal,Rank_Equal,Rank_Diff
0,PALO ALTO CA / HAMILTON #2,23682,0.723298,1,0.672263,2,1
1,PALO ALTO CA / WEBSTER #1,14408,0.647613,2,0.674869,1,1
2,PALO ALTO CA / WEBSTER #2,12654,0.604583,3,0.647787,3,0
3,PALO ALTO CA / HAMILTON #1,14836,0.512089,4,0.555134,9,5
4,PALO ALTO CA / HIGH #4,13844,0.506990,5,0.554710,10,5


,Station Name,C1_so_phien,Score_Entropy,Rank_Entropy,Score_Equal,Rank_Equal,Rank_Diff
28,PALO ALTO CA / CAMBRIDGE #5,4551,0.196495,29,0.419360,28,1
29,PALO ALTO CA / CAMBRIDGE #3,4472,0.189231,30,0.419249,29,1
30,PALO ALTO CA / MPL #1,1044,0.187964,31,0.280276,32,1
31,PALO ALTO CA / MPL #3,1420,0.179308,32,0.281993,31,1
32,PALO ALTO CA / CAMBRIDGE #4,3426,0.170144,33,0.404260,30,3


Baseline Spearman: 0.8191844919786097
Baseline Kendall: 0.6363636363636364
Mean/max rank difference: 4.303030303030303 14


## 5. Đánh giá Tính Bền vững Thứ hạng qua 7 Kịch bản Độ nhạy (S0 – S6)

### 🎯 Mục tiêu Phân tích Độ nhạy (Sensitivity Analysis):
Kiểm tra xem thứ bậc của các trạm có bị phụ thuộc thái quá vào phương pháp tính trọng số Entropy hay không, thông qua 7 kịch bản đối sánh:
- **S0 (Kịch bản cơ sở):** Trọng số Entropy khách quan.
- **S1 (Trọng số đều - Equal Weights):** Gán trọng số bằng nhau $w_j = 1/6 \approx 0.1667$ cho cả 6 tiêu chí (đóng vai trò đối chứng khách quan).
- **S2 (Loại bỏ $C_1$):** Bỏ tiêu chí số phiên sạc để triệt tiêu ảnh hưởng đa cộng tuyến với $C_2$ ($r = 0.982$).
- **S3 (Loại bỏ $C_2$):** Bỏ tiêu chí tổng kWh điện năng.
- **S4 (Tăng trọng số Chất lượng):** Tăng gấp đôi trọng số cho nhóm công suất và tỷ lệ sạc có ích ($C_3, C_4$).
- **S5 (Tăng trọng số Chống chiếm dụng):** Tăng gấp đôi trọng số cho nhóm kiểm soát nhàn rỗi ($C_5, C_6$).
- **S6 (Loại bỏ nhóm Tiêu chí Chi phí):** Chỉ xếp hạng dựa trên 4 tiêu chí lợi ích ($C_1 \to C_4$).

### 🔍 Phát hiện Nghiệp vụ Then chốt:
- Trạm đứng đầu **HAMILTON #2** và cụm **WEBSTER #1, #2** giữ vững vị trí trong Top đầu qua hầu hết các kịch bản.
- Kịch bản S2 (bỏ $C_1$) làm 7 trạm đổi từ 5 bậc trở lên, chứng minh quy mô số phiên đóng vai trò phân hóa mạnh trong mạng lưới.

In [5]:
df_sensitivity = run_sensitivity(X, NV4_CRITERIA)
display(df_sensitivity)
rank_robustness = summarize_rank_robustness(X, NV4_CRITERIA, df_sensitivity)
display(pd.Series({k: v for k, v in rank_robustness.items() if k != "scenarios"}, name="value").to_frame())
display(pd.DataFrame(rank_robustness["scenarios"]).T)


,Scenario,Criteria_Count,Spearman_Entropy,Kendall_Entropy,Mean_Rank_Diff_Entropy,Max_Rank_Diff_Entropy,Stations_Diff_ge5_Entropy,Top5_Overlap_Entropy,Spearman_Equal,Kendall_Equal,Mean_Rank_Diff_Equal,Max_Rank_Diff_Equal,Stations_Diff_ge5_Equal,Top5_Overlap_Equal
0,S0: Gốc (C1-C6),6,1.0000,1.0000,0.00,0,0,5/5,1.0000,1.0000,0.00,0.0,0.0,5/5
1,S1: Loại C2 (Tổng kWh),5,0.9559,0.8447,2.00,8,3,4/5,0.9178,0.8030,2.85,9.0,8.0,4/5
2,S2: Loại C1 (Số phiên),5,0.9261,0.7727,2.85,11,7,3/5,0.8927,0.7803,3.15,12.0,9.0,4/5
3,S3: Loại C6 (Chiếm dụng nặng),5,0.9970,0.9811,0.30,2,0,5/5,0.9890,0.9432,0.73,5.0,1.0,5/5
4,S4: Loại C4 (Thời gian sạc),5,0.9920,0.9432,0.85,3,0,5/5,0.9208,0.7841,2.55,10.0,6.0,4/5
5,S5: C5 trọng số thời lượng,6,0.9987,0.9886,0.18,2,0,5/5,0.9923,0.9508,0.73,4.0,0.0,5/5
6,S6: Vector Normalization,6,0.8870,0.7311,3.09,16,6,5/5,NaN,NaN,NaN,NaN,NaN,N/A


,value
station_count,33
correlation_c1_c2,0.981503
correlation_c5_c6,0.985274
entropy_weight_c1_c2_sum,0.572064
interpretation,"Ranking is conditional on criteria, weights an..."


,spearman_entropy,stations_shifted_at_least_5_ranks,max_rank_shift,top5_overlap
S1,0.9559,3,8,4/5
S2,0.9261,7,11,3/5
S6,0.887,6,16,5/5


## 6. Lưu trữ Kết quả Thực nghiệm & Kiểm toán Metadata

### 🎯 Mục tiêu của Bước:
Lưu trữ toàn bộ các bảng kết quả tính toán từ cùng một lượt chạy vào thư mục `outputs/` nhằm phục vụ báo cáo và đảm bảo khả năng tái lập hoàn toàn (*Reproducibility*):
1. `nv4_session_reconciliation.csv`: Sổ cái đối soát phễu dữ liệu tuần tự.
2. `nv4_station_criteria.csv`: Ma trận 33 trạm × 6 tiêu chí vận hành thực tế.
3. `nv4_criteria_weights.csv`: Bảng đối chiếu trọng số Entropy vs Trọng số đều.
4. `nv4_topsis_rankings.csv`: Bảng xếp hạng chi tiết 33 trạm kèm điểm số $CC_i$, khoảng cách $D^+$, $D^-$.
5. `nv4_sensitivity_analysis.csv`: Bảng biến động thứ hạng qua 7 kịch bản S0 – S6.
6. `nv4_run_metadata.json`: Lưu vết thời gian chạy, số phiên hợp lệ và checksum kỹ thuật.

In [6]:
reconciliation.to_csv(OUTPUT_DIR / "nv4_session_reconciliation.csv", index=False, encoding="utf-8-sig")
pd.Series(run_metadata, name="value").to_json(OUTPUT_DIR / "nv4_run_metadata.json", indent=2, force_ascii=False)
X[criterion_columns].to_csv(OUTPUT_DIR / "nv4_station_criteria.csv", encoding="utf-8-sig")
entropy_table.to_csv(OUTPUT_DIR / "nv4_criteria_weights.csv", encoding="utf-8-sig")
df_rank.to_csv(OUTPUT_DIR / "nv4_topsis_rankings.csv", index=False, encoding="utf-8-sig")
corr_pearson.to_csv(OUTPUT_DIR / "nv4_criteria_correlations_pearson.csv", encoding="utf-8-sig")
corr_spearman.to_csv(OUTPUT_DIR / "nv4_criteria_correlations_spearman.csv", encoding="utf-8-sig")
df_sensitivity.to_csv(OUTPUT_DIR / "nv4_sensitivity_analysis.csv", index=False, encoding="utf-8-sig")
print("Saved NV4 tables from this run to", OUTPUT_DIR)
(OUTPUT_DIR / "nv4_rank_robustness.json").write_text(json.dumps(rank_robustness, ensure_ascii=False, indent=2), encoding="utf-8")


Saved NV4 tables from this run to /home/kedora/SEM5/ba/project/ev-charging-analytics---IS403/outputs


778